# 01: GCP 実機ハンズオン — PyIceberg × BigLake Metastore × BigQuery で体験する自動スキーマ進化・Upsert・タイムトラベル

## このノートブックで学習できること

- GCS 上に配置した CSV ファイルを PyIceberg から BigLake Metastore（Apache Iceberg REST Catalog）経由で Lakehouse バケット（GCS）へ Iceberg テーブルとして出力し、既存の BigQuery データセット配下から直接 `SELECT` できる構成
- 1 回目の連携: `id, name, age` の 3 カラム・100 行の CSV を取り込み、BigQuery で 100 行・3 カラムとして参照できることの確認
- 2 回目の連携: 既存 100 行（同じ ID）と新規 10 行の計 110 行で、3 列目に `prefecture` カラムが追加された CSV（`id, name, prefecture, age`）を、PyIceberg 側のコードを変更せず同じ関数で取り込む手順
- 3 回目の連携: `prefecture` カラムが欠損した追加 10 行の CSV（`id, name, age`）を同じ関数で取り込んだとき、テーブルから `prefecture` 列が削除されず、欠損列が `NULL` として安全にロードされることの確認
- BigQuery の `FOR SYSTEM_TIME AS OF` を使い、1 回目・2 回目・3 回目の各ロード時点の断面（タイムトラベル）を参照・比較する手順
- Iceberg の `field-id` ベースのスキーマ管理、スナップショット情報、GCS 上の物理ファイル（Parquet / `metadata.json` / Manifest）の確認

## 到達ゴール

- 上流 CSV の途中列へのカラム追加や、逆に一部カラムが欠損したデータの流入が起きても、取り込みコードの修正や BigQuery 側の `ALTER TABLE` なしで安全にデータ連携を継続し、過去時点の断面も参照できる仕組みを実機で再現・説明できる

## 事前準備

詳細は [GCP 実機ハンズオン手順書](../docs/handson/01_gcp_pyiceberg_biglake_handson.md) を参照してください。事前に以下の 3 点が完了している前提です。

1. `.env.example` をコピーして `.env` を作成し、`GCP_PROJECT_ID`・`GCS_LAKEHOUSE_BUCKET`・`BQ_DATASET`（既存の BigQuery データセット名）を設定済み
2. `gcloud auth application-default login`（ADC 認証済み）
3. `gcloud services enable biglake.googleapis.com storage.googleapis.com bigquery.googleapis.com`


## 1. 環境設定と BigLake カタログへの接続

### 1.1 環境変数（`.env`）の読み込みと設定値の確認

まず、リポジトリ直下の `.env` から GCP プロジェクト ID やバケット名などの設定を読み込みます。コード内に環境固有の値をハードコードせず、環境変数経由で参照する構成です。

> **Note（バケット設計について）**: 本番環境では Landing バケットと Lakehouse バケットを物理的に分離するのがベストプラクティスです（[第4章: GCS バケット設計](../docs/guides/04_gcs_design_and_security.md)）。手軽に試す場合は `.env` で `GCS_LANDING_BUCKET` と `GCS_LAKEHOUSE_BUCKET` に同じバケットを指定し、プレフィックスで分離して実行できます。


In [ ]:
from datetime import datetime, timezone
import io
import os
from pathlib import Path
import random
import textwrap
import time

from dotenv import load_dotenv
import google.auth
from google.auth.transport.requests import Request
from google.cloud import bigquery, storage
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pv
from pyiceberg.catalog.rest import RestCatalog
from pyiceberg.expressions import In

# リポジトリ直下の .env を読み込み（未作成の場合は .env.example をコピーしてください）
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT_DIR / ".env")

PROJECT_ID = os.environ["GCP_PROJECT_ID"]
LOCATION = os.environ.get("GCP_LOCATION", "asia-northeast1")
LAKEHOUSE_BUCKET = os.environ["GCS_LAKEHOUSE_BUCKET"]
LANDING_BUCKET = os.environ.get("GCS_LANDING_BUCKET", LAKEHOUSE_BUCKET)
LANDING_PREFIX = os.environ.get("GCS_LANDING_PREFIX", "handson_schema_evolution/landing")

BQ_DATASET = os.environ["BQ_DATASET"]
TABLE_NAME = os.environ.get("ICEBERG_TABLE_NAME", "users_master")
TABLE_ID = f"{BQ_DATASET}.{TABLE_NAME}"
TABLE_LOCATION = f"gs://{LAKEHOUSE_BUCKET}/{BQ_DATASET}/{TABLE_NAME}"
BQ_TABLE_FQN = f"`{PROJECT_ID}.{BQ_DATASET}.{TABLE_NAME}`"

print(f"LOCATION       : {LOCATION}")
print(f"LANDING_PREFIX : {LANDING_PREFIX}")
print(f"TABLE_ID       : {TABLE_ID}")
print(f"TABLE_LOCATION : {TABLE_LOCATION}")
print(f"BQ_TABLE_FQN   : {BQ_TABLE_FQN}")


環境変数の読み込みが完了し、対象の BigQuery データセット配下のテーブル識別子（`TABLE_ID`）、GCS 出力先（`TABLE_LOCATION`）、BigQuery 参照名（`BQ_TABLE_FQN`）が確定しました。

### 1.2 BigLake Iceberg REST Catalog への接続とハンズオン用テーブルの初期化

次に、ADC（Application Default Credentials）を使って BigLake Metastore の Iceberg REST Catalog エンドポイント（`https://biglake.googleapis.com/iceberg/v1/restcatalog`）へ接続します。`warehouse` に `bq://projects/<PROJECT_ID>` を指定することで、既存の BigQuery データセット（`BQ_DATASET`）をそのまま Namespace として利用し、その配下に Iceberg テーブルを作成できます。


In [ ]:
def get_biglake_catalog() -> RestCatalog:
    """ADC を使って BigLake Metastore (Apache Iceberg REST Catalog) へ接続する。"""
    creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
    creds.refresh(Request())
    return RestCatalog(
        name="biglake",
        **{
            "uri": "https://biglake.googleapis.com/iceberg/v1/restcatalog",
            "warehouse": f"bq://projects/{PROJECT_ID}",
            "token": creds.token,
            "header.x-goog-user-project": PROJECT_ID,
            "rest.sigv4-enabled": "false",
        },
    )


storage_client = storage.Client(project=PROJECT_ID)
bq_client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
catalog = get_biglake_catalog()

if (BQ_DATASET,) not in catalog.list_namespaces():
    catalog.create_namespace(BQ_DATASET)
    print(f"データセット（Namespace）を新規作成しました: {BQ_DATASET}")

if catalog.table_exists(TABLE_ID):
    catalog.drop_table(TABLE_ID)
    print(f"既存テーブル {TABLE_ID} を初期化のため削除しました。")

print("BigLake Catalog 接続および初期化が完了しました。")
print("対象データセット確認:", (BQ_DATASET,) in catalog.list_namespaces())


BigLake Iceberg REST Catalog への接続と対象データセットの準備ができました。

---

## 2. コードを変更せず使い回す共通関数の定義

### 2.1 ヘルパー関数（GCS 読込・名前ベースのスキーマ整列・BigQuery 実行）

ここでは 3 つの小さなヘルパー関数を定義します。`align_to_table_schema()` は、CSV の列順が途中挿入によって変わったり、逆に一部カラムが欠損していたりしても、カラム名ベースで Iceberg テーブルの最新スキーマに自動整列（欠損列は `NULL` で補完）する役割を持ちます。


In [ ]:
def read_csv_from_gcs(bucket_name: str, blob_path: str) -> pa.Table:
    """GCS 上の CSV ファイルを読み込み、PyArrow Table を返す。"""
    blob = storage_client.bucket(bucket_name).blob(blob_path)
    csv_bytes = blob.download_as_bytes()
    return pv.read_csv(io.BytesIO(csv_bytes))


def align_to_table_schema(arrow_tbl: pa.Table, iceberg_tbl) -> pa.Table:
    """入力 Arrow Table の列を、名前ベースで Iceberg テーブルの最新スキーマに整列・型変換する。"""
    target_schema = iceberg_tbl.schema().as_arrow()
    arrays = []
    for field in target_schema:
        if field.name in arrow_tbl.column_names:
            arrays.append(arrow_tbl[field.name].cast(field.type))
        else:
            arrays.append(pa.nulls(arrow_tbl.num_rows, type=field.type))
    return pa.Table.from_arrays(arrays, schema=target_schema)


def query_bigquery(sql: str, show_sql: bool = True) -> pd.DataFrame:
    """BigQuery で SQL を実行し、実行クエリを表示したうえで結果を pandas DataFrame で返す。"""
    formatted_sql = textwrap.dedent(sql).strip()
    if show_sql:
        print("--- [BigQuery 実行クエリ] ---")
        print(formatted_sql)
        print("-----------------------------")
    job_config = bigquery.QueryJobConfig(labels={"datacloud": "antigravity"})
    return bq_client.query(formatted_sql, job_config=job_config).to_dataframe()


print("ヘルパー関数を定義しました。")


### 2.2 共通取り込み関数 `ingest_csv_to_iceberg()` の定義

この関数には `name`, `age`, `prefecture` といった具体的なカラム名が一切書かれていません。1 回目（3 列）・2 回目（4 列）・3 回目（`prefecture` 欠損の 3 列）のすべてで、次の 4 ステップを同じコードで実行します。

1. GCS 上の CSV を PyArrow Table として読み込む
2. テーブルが未作成なら新規作成し、既存なら `update_schema().union_by_name()` で新カラムを自動マージする（入力に無い既存カラムは削除されず維持される）
3. `align_to_table_schema()` で入力列を最新スキーマの列順に整列する（入力に欠損している列は `NULL` で自動補完）
4. `overwrite(aligned, overwrite_filter=In(key_col, keys))` で既存 ID を置換しつつ新規 ID を追加する（キーベース Upsert）


In [ ]:
def ingest_csv_to_iceberg(
    bucket_name: str,
    blob_path: str,
    table_id: str,
    table_location: str = TABLE_LOCATION,
    key_col: str = "id",
):
    """GCS の CSV を読み込み、自動スキーマ進化 + キーベース Upsert で Iceberg テーブルへ反映する。"""
    cat = get_biglake_catalog()
    incoming = read_csv_from_gcs(bucket_name, blob_path)
    print(f"[1/4 読込] gs://{bucket_name}/{blob_path} -> {incoming.num_rows} 行, 列順: {incoming.column_names}")

    if not cat.table_exists(table_id):
        tbl = cat.create_table(table_id, schema=incoming.schema, location=table_location)
        print(f"[2/4 作成] 新規 Iceberg テーブルを作成しました: {table_id} (location={table_location})")
    else:
        tbl = cat.load_table(table_id)
        before_cols = [f.name for f in tbl.schema().fields]
        with tbl.update_schema() as update:
            update.union_by_name(incoming.schema)
        tbl.refresh()
        after_cols = [f.name for f in tbl.schema().fields]
        if before_cols != after_cols:
            added = [c for c in after_cols if c not in before_cols]
            print(f"[2/4 スキーマ進化] 新カラムを自動追加しました: {added} -> {after_cols}")
        else:
            missing = [c for c in after_cols if c not in incoming.column_names]
            if missing:
                print(f"[2/4 スキーマ維持] 入力に欠損列 {missing} がありますが、テーブル定義は維持します: {after_cols}")
            else:
                print(f"[2/4 スキーマ確認] 変更なし: {after_cols}")

    aligned = align_to_table_schema(incoming, tbl)
    print(f"[3/4 列整列] テーブルスキーマに合わせて整列しました: {aligned.column_names}")

    keys = aligned[key_col].to_pylist()
    tbl.overwrite(aligned, overwrite_filter=In(key_col, keys))
    tbl.refresh()

    snap = tbl.current_snapshot()
    total_rows = tbl.scan().to_arrow().num_rows
    print(f"[4/4 完了] Upsert 完了 | snapshot_id={snap.snapshot_id} | テーブル総行数={total_rows} 行")
    return tbl


# 後段のタイムトラベル確認用に、各バッチ完了時点の (UTC タイムスタンプ, その時点のカラム一覧) を記録する辞書
load_checkpoints: dict[str, tuple[str, list[str]]] = {}
print("共通取り込み関数 ingest_csv_to_iceberg() を定義しました。")


共通関数の準備が整いました。ここから 1 回目・2 回目・3 回目のデータ連携を順番に実行して挙動を確認します。

---

## 3. 【1回目】3カラム・100行の CSV 連携と BigQuery 参照

### 3.1 1回目の CSV（`id, name, age`・100行）を作成して GCS に配置する

まず、`id`（1〜100）、`name`、`age` の 3 カラム・100 行の CSV ファイルを作成し、GCS の Landing パスへアップロードします。


In [ ]:
rng = random.Random(42)

batch1_df = pd.DataFrame({
    "id": list(range(1, 101)),
    "name": [f"User_{i:03d}" for i in range(1, 101)],
    "age": [rng.randint(20, 65) for _ in range(100)],
})

batch1_blob_path = f"{LANDING_PREFIX}/batch1_users.csv"
storage_client.bucket(LANDING_BUCKET).blob(batch1_blob_path).upload_from_string(
    batch1_df.to_csv(index=False),
    content_type="text/csv",
)

print(f"1回目 CSV を GCS へアップロードしました: gs://{LANDING_BUCKET}/{batch1_blob_path}")
print(f"行数: {len(batch1_df)} 行 / 列順: {list(batch1_df.columns)}")
display(batch1_df.head(5))


1 回目の CSV（`id, name, age` の 3 列・100 行）が GCS 上に配置されました。

### 3.2 PyIceberg で Lakehouse バケットへ出力する

次に、`ingest_csv_to_iceberg()` を呼び出して、GCS 上の CSV を Lakehouse バケットへ Iceberg テーブルとして出力します。初回のためテーブルが自動作成されます。後でタイムトラベル確認に使うため、完了時点の UTC タイムスタンプも記録しておきます。


In [ ]:
tbl = ingest_csv_to_iceberg(LANDING_BUCKET, batch1_blob_path, TABLE_ID)

time.sleep(2)
ts1 = datetime.now(timezone.utc).isoformat()
cols1 = [f.name for f in tbl.schema().fields]
load_checkpoints["1回目 (100行 / 3列)"] = (ts1, cols1)
print(f"記録したタイムスタンプ (UTC): {ts1} | カラム: {cols1}")
print("\n--- 1回目連携後の Iceberg スキーマ ---")
print(tbl.schema())


出力結果から、新規 Iceberg テーブルが作成され、`1: id`, `2: name`, `3: age` の 3 つの `field-id` が採番されたこと、および 100 行が書き込まれたことが確認できます。

### 3.3 BigQuery から 1 回目の結果を直接クエリする

BigLake Iceberg REST Catalog（`bq://projects/<PROJECT_ID>`）を利用しているため、BigQuery の既存データセット内にテーブルが直接登録されます。通常の BigQuery テーブルと同じ 3 パート名（`` `project.dataset.table` ``）でそのまま SQL を実行できます。


In [ ]:
bq_count_1 = query_bigquery(f"SELECT COUNT(*) AS row_count FROM {BQ_TABLE_FQN}")
bq_sample_1 = query_bigquery(f"SELECT * FROM {BQ_TABLE_FQN} ORDER BY id LIMIT 5")

print("=== BigQuery クエリ結果（1回目連携後） ===")
print(f"行数   : {bq_count_1['row_count'].iloc[0]} 行")
print(f"カラム : {list(bq_sample_1.columns)}")
display(bq_sample_1)


BigQuery から `100 行`・3 カラム（`['id', 'name', 'age']`）として参照できました。

---

## 4. 【2回目】3列目に `prefecture` を追加した 110 行の CSV をコード変更なしで連携する

### 4.1 2回目の CSV（`id, name, prefecture, age`・110行）を作成して GCS に配置する

上流システムの仕様変更を想定し、次の特徴を持つ 2 回目の CSV を作成して GCS へアップロードします。

- 行数: 1 回目と同じ `id = 1..100` の 100 行 ＋ 新規 `id = 101..110` の 10 行 ＝ 計 110 行
- カラム変更: 末尾ではなく 3 列目に `prefecture` を挿入（列順: `id, name, prefecture, age`）


In [ ]:
PREFECTURES = ["東京都", "大阪府", "愛知県", "福岡県", "北海道", "神奈川県", "京都府", "宮城県"]

# 既存100行（同じID）の age はそのまま維持しつつ、新規10行を追加
ages_110 = list(batch1_df["age"]) + [rng.randint(20, 65) for _ in range(10)]

batch2_df = pd.DataFrame({
    "id": list(range(1, 111)),
    "name": [f"User_{i:03d}" for i in range(1, 111)],
    "prefecture": [rng.choice(PREFECTURES) for _ in range(110)],  # 3列目に挿入
    "age": ages_110,
})

batch2_blob_path = f"{LANDING_PREFIX}/batch2_users.csv"
storage_client.bucket(LANDING_BUCKET).blob(batch2_blob_path).upload_from_string(
    batch2_df.to_csv(index=False),
    content_type="text/csv",
)

print(f"2回目 CSV を GCS へアップロードしました: gs://{LANDING_BUCKET}/{batch2_blob_path}")
print(f"行数: {len(batch2_df)} 行 / 列順: {list(batch2_df.columns)}")
display(batch2_df.iloc[[0, 1, 99, 100, 109]])


2 回目の CSV では列順が `['id', 'name', 'prefecture', 'age']` となっており、`id = 1..100` が既存、`id = 101..110` が新規行です。

### 4.2 コード変更なしで同じ `ingest_csv_to_iceberg()` を実行する

1 回目と全く同じ関数を、引数の CSV パスだけ変えて実行します。取り込み側のコードは 1 行も変更しません。


In [ ]:
tbl = ingest_csv_to_iceberg(LANDING_BUCKET, batch2_blob_path, TABLE_ID)

time.sleep(2)
ts2 = datetime.now(timezone.utc).isoformat()
cols2 = [f.name for f in tbl.schema().fields]
load_checkpoints["2回目 (110行 / 4列)"] = (ts2, cols2)
print(f"記録したタイムスタンプ (UTC): {ts2} | カラム: {cols2}")


ログの 4 ステップから、新カラム `prefecture` が `union_by_name()` で自動追加され、名前ベースで整列されたうえで既存 100 行の更新＋新規 10 行の追加（総行数 `110 行`）が完了したことがわかります。

### 4.3 BigQuery から更新結果を確認する（`ALTER TABLE` 不要）

BigQuery 側ではスキーマ変更の DDL を一切実行していません。1 回目と同じテーブルに対してクエリを投げ、行数と既存行・新規行の内容を確認します。


In [ ]:
bq_count_2 = query_bigquery(f"SELECT COUNT(*) AS row_count FROM {BQ_TABLE_FQN}")
bq_sample_2 = query_bigquery(
    f"SELECT * FROM {BQ_TABLE_FQN} WHERE id IN (1, 2, 100, 101, 110) ORDER BY id"
)

print("=== BigQuery クエリ結果（2回目連携後） ===")
print(f"行数   : {bq_count_2['row_count'].iloc[0]} 行（既存100行が更新 + 新規10行が追加）")
print(f"カラム : {list(bq_sample_2.columns)}")
display(bq_sample_2)


BigQuery 側で何の変更もしていないのに、`110 行`・4 カラム（`id, name, age, prefecture`）として即座に参照できました。

### 4.4 BigQuery で新カラム `prefecture` を使った集計クエリを実行する

自動追加された `prefecture` カラムが、BigQuery 上でそのまま `GROUP BY` 集計に使えることも確認します。


In [ ]:
bq_by_pref = query_bigquery(
    f"SELECT prefecture, COUNT(*) AS cnt FROM {BQ_TABLE_FQN} GROUP BY prefecture ORDER BY cnt DESC"
)
display(bq_by_pref)


新カラム `prefecture` 別の件数集計がそのまま実行できました。

---

## 5. 【3回目】`prefecture` カラムが欠損した追加 10 行の CSV を連携する（欠損列の NULL 補完）

### 5.1 3回目の CSV（`id = 111..120`・`prefecture` 欠損の 3 列・10 行）を作成して GCS に配置する

今度は逆に、旧フォーマットの別経路や一時的な連携不備を想定し、**`prefecture` カラムが存在しない 3 カラム（`id, name, age`）の追加 10 レコード（`id = 111..120`）** を GCS にアップロードします。


In [ ]:
batch3_df = pd.DataFrame({
    "id": list(range(111, 121)),
    "name": [f"User_{i:03d}" for i in range(111, 121)],
    "age": [rng.randint(20, 65) for _ in range(10)],
})

batch3_blob_path = f"{LANDING_PREFIX}/batch3_users_missing_pref.csv"
storage_client.bucket(LANDING_BUCKET).blob(batch3_blob_path).upload_from_string(
    batch3_df.to_csv(index=False),
    content_type="text/csv",
)

print(f"3回目 CSV を GCS へアップロードしました: gs://{LANDING_BUCKET}/{batch3_blob_path}")
print(f"行数: {len(batch3_df)} 行 / 列順: {list(batch3_df.columns)}（prefecture 欠損）")
display(batch3_df.head(5))


3 回目の CSV には `prefecture` 列が含まれていません。

### 5.2 コード変更なしで同じ `ingest_csv_to_iceberg()` を実行する

ここでも 1 回目・2 回目と全く同じ関数をそのまま実行します。


In [ ]:
tbl = ingest_csv_to_iceberg(LANDING_BUCKET, batch3_blob_path, TABLE_ID)

time.sleep(2)
ts3 = datetime.now(timezone.utc).isoformat()
cols3 = [f.name for f in tbl.schema().fields]
load_checkpoints["3回目 (120行 / 欠損列NULL)"] = (ts3, cols3)
print(f"記録したタイムスタンプ (UTC): {ts3} | カラム: {cols3}")


`[2/4 スキーマ維持]` のログの通り、`prefecture` 列はテーブルから削除されず維持されたまま、追加 10 行が取り込まれて総行数が `120 行` になりました。

### 5.3 BigQuery でカラム維持と欠損列の `NULL` ロードを確認する

BigQuery から集計とサンプル行を確認し、既存の `id = 1..110` には `prefecture` が残ったまま、3 回目で追加された `id = 111..120` の 10 行だけ `prefecture` が `NULL` になっていることを検証します。


In [ ]:
bq_check_3 = query_bigquery(f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT(prefecture) AS prefecture_not_null,
  COUNTIF(prefecture IS NULL) AS prefecture_is_null
FROM {BQ_TABLE_FQN}
""")
bq_sample_3 = query_bigquery(
    f"SELECT * FROM {BQ_TABLE_FQN} WHERE id IN (100, 110, 111, 115, 120) ORDER BY id"
)

print("=== BigQuery クエリ結果（3回目連携後） ===")
print(f"カラム: {list(bq_sample_3.columns)}（prefecture は削除されず維持）")
display(bq_check_3)
display(bq_sample_3)


`total_rows = 120`、`prefecture_not_null = 110`、`prefecture_is_null = 10` となり、カラムが消えることなく欠損列が `NULL` として安全にロードされたことが確認できました。

---

## 6. 各ロード時点の断面を確認する（BigQuery タイムトラベル）

### 6.1 `FOR SYSTEM_TIME AS OF` で 1 回目・2 回目・3 回目の断面を一括比較する

BigQuery の標準 SQL 構文 `FOR SYSTEM_TIME AS OF TIMESTAMP(...)` を使い、先ほど記録した 3 つの時点それぞれで同じテーブルをクエリします。各ロード時点の行数・カラム構成・`prefecture` の NULL 件数を一覧で比較します。

> **Note（スキーマ進化前の時点に対する `SELECT *` の注意点）**: BigQuery で Iceberg テーブルをタイムトラベル参照する際、1 回目のように「現在のスキーマにはあるが当時まだ存在しなかった列（`prefecture`）」を `SELECT *` で読み込もうとすると `Missing column: 'prefecture'` エラーになります。そのため、各時点で存在していたカラムを明示して `SELECT` するのが実務上のポイントです。


In [ ]:
snapshots_summary = []
snapshot_dfs: dict[str, pd.DataFrame] = {}

for label, (ts_utc, cols) in load_checkpoints.items():
    col_sql = ", ".join(cols)
    df_at_t = query_bigquery(
        f"SELECT {col_sql} FROM {BQ_TABLE_FQN} FOR SYSTEM_TIME AS OF TIMESTAMP('{ts_utc}') ORDER BY id"
    )
    snapshot_dfs[label] = df_at_t
    pref_nulls = int(df_at_t["prefecture"].isna().sum()) if "prefecture" in df_at_t.columns else None
    snapshots_summary.append({
        "時点": label,
        "timestamp_utc": ts_utc,
        "行数": len(df_at_t),
        "カラム数": len(df_at_t.columns),
        "カラム一覧": ", ".join(df_at_t.columns),
        "prefecture_NULL件数": pref_nulls,
    })

display(pd.DataFrame(snapshots_summary))


同じ 1 つの Iceberg テーブルに対して、1 回目（100 行・3 列）・2 回目（110 行・4 列・NULL 0 件）・3 回目（120 行・4 列・NULL 10 件）の各時点の断面が、BigQuery の SQL だけでそのまま再現できました。

### 6.2 各時点の末尾レコードを見比べる

それぞれの断面で末尾 3 行を表示し、各ロード時点でどこまでのデータが存在していたかを視覚的に確認します。


In [ ]:
for label, df_at_t in snapshot_dfs.items():
    print(f"=== {label} の末尾 3 行 ===")
    display(df_at_t.tail(3))


1 回目は `id = 98..100`（`prefecture` 列なし）、2 回目は `id = 108..110`（`prefecture` あり）、3 回目は `id = 118..120`（`prefecture = None`）となっており、障害調査や過去時点の再現分析にそのまま活用できることが確認できました。

---

## 7. Iceberg の内部構造を検証する

### 7.1 スキーマの `field-id` を確認する

なぜ 2 回目で 3 列目に `prefecture` が割り込んでも `age` の値がずれず、3 回目で `prefecture` が欠損してもスキーマが壊れなかったのかを Iceberg のスキーマ定義から確認します。


In [ ]:
print(tbl.schema())


`age` は 1 回目で採番された不変の `field-id = 3` を維持しており、2 回目で追加された `prefecture` は `field-id = 4`（optional）として保持されています。Iceberg は列の位置ではなく `field-id` で列を識別するため、途中列への追加にも欠損列にも安全に対応できます。

### 7.2 スナップショット情報を確認する

現在のテーブルが指しているスナップショットの統計情報（レコード数やデータファイル数）を確認します。


In [ ]:
for idx, snap in enumerate(tbl.snapshots(), start=1):
    summary = dict(snap.summary.additional_properties) if snap.summary else {}
    print(
        f"Snapshot {idx}: id={snap.snapshot_id} | op={snap.summary.operation if snap.summary else None} "
        f"| total-records={summary.get('total-records')} | added-data-files={summary.get('added-data-files')}"
    )


スナップショットのサマリに `total-records=120` が記録されており、カタログ経由で全件スキャンせずに行数などの統計情報を把握できることがわかります。

### 7.3 GCS Lakehouse バケット上の物理ファイル一覧を確認する

最後に、GCS の Lakehouse バケット配下に実際にどのようなファイルが生成されたかを確認します。


In [ ]:
table_prefix = tbl.location().replace(f"gs://{LAKEHOUSE_BUCKET}/", "")
blobs = list(storage_client.list_blobs(LAKEHOUSE_BUCKET, prefix=table_prefix))

print(f"生成されたファイル総数: {len(blobs)} 件")
for b in blobs:
    rel_path = b.name.replace(table_prefix, "").lstrip("/")
    print(f"  {rel_path} ({b.size:,} bytes)")


`data/` 配下に 1〜3 回目の Parquet ファイル、`metadata/` 配下に世代ごとの `metadata.json` と Manifest ファイル（`.avro`）がオープンフォーマットとして保存されていることが確認できました。

### まとめ

- Iceberg は不変の `field-id` でカラムを管理するため、CSV の途中（3 列目）に新カラムが追加されても、逆に一部カラムが欠損したデータが流入しても、取り込みコードを変更せずに安全にロードを継続できます。
- BigLake Metastore（Iceberg REST Catalog、`warehouse=bq://projects/<PROJECT_ID>`）を介することで、PyIceberg がコミットした最新のスキーマとデータがそのまま BigQuery に反映され、`FOR SYSTEM_TIME AS OF` による過去断面のタイムトラベル参照も標準 SQL で実行できます。
